In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading Test Data and Best Model (XGBoost)...")
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

model = joblib.load('../models/xgboost.pkl')
print("Predicting on Test Set...")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print(f"\n--- Test Set Performance ---")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

os.makedirs('../visualizations/evaluation', exist_ok=True)
with open('../models/final_test_metrics.json', 'w') as f:
    json.dump({'roc_auc': float(roc_auc), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'accuracy': float(acc)}, f, indent=4)

In [ ]:
# Visualization 1: Detailed Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Active (0)', 'Churned (1)'], 
            yticklabels=['Active (0)', 'Churned (1)'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title(f'Final Confusion Matrix (Test Set)\nTrue Positives: {cm[1,1]} | False Positives: {cm[0,1]}')
plt.savefig('../visualizations/evaluation/final_confusion_matrix.png')
plt.close()
print("Saved final_confusion_matrix.png")

In [ ]:
# Visualization 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.savefig('../visualizations/evaluation/roc_curve.png')
plt.close()
print("Saved roc_curve.png")

In [ ]:
# Visualization 3: Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='purple', lw=2, label=f'PR curve (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.savefig('../visualizations/evaluation/precision_recall_curve.png')
plt.close()
print("Saved precision_recall_curve.png")

In [ ]:
# Visualization 4: Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 8))
sns.barplot(x=importances[indices][:15], y=np.array(X_test.columns)[indices][:15])
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.savefig('../visualizations/evaluation/feature_importance.png')
plt.close()
print("Saved feature_importance.png")

In [ ]:
# Visualization 5: Prediction Distribution
plt.figure(figsize=(10, 6))
sns.histplot(y_prob[y_test == 0], color='blue', label='Actual Active', kde=True, stat="density", linewidth=0, alpha=0.5)
sns.histplot(y_prob[y_test == 1], color='red', label='Actual Churn', kde=True, stat="density", linewidth=0, alpha=0.5)
plt.title('Prediction Probability Distribution by Actual Class')
plt.xlabel('Predicted Probability of Churn')
plt.legend()
plt.savefig('../visualizations/evaluation/prediction_distribution.png')
plt.close()
print("Saved prediction_distribution.png")

In [ ]:
# Visualization 6: Calibration Curve
prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=1, label='XGBoost')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly calibrated')
plt.title('Calibration Curve of Probabilities')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.savefig('../visualizations/evaluation/calibration_curve.png')
plt.close()
print("Saved calibration_curve.png")

In [ ]:
# Section 4: Error Analysis
results_df = X_test.copy()
results_df['Actual'] = y_test
results_df['Predicted'] = y_pred
results_df['Probability'] = y_prob

false_positives = results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)]
false_negatives = results_df[(results_df['Actual'] == 1) & (results_df['Predicted'] == 0)]

print(f"\nTotal False Positives: {len(false_positives)}")
print(f"Total False Negatives: {len(false_negatives)}")

# Q1 & Q2: Feature characteristics
# Notice if Recency or Purchases_Last30Days were somehow confusing the model
print("\n--- False Negative Characteristics (Missed Churners) ---")
if len(false_negatives) > 0:
    print(false_negatives[['Recency', 'Frequency', 'Probability']].describe())

print("\n--- False Positive Characteristics (Wrongly Flagged as Churners) ---")
if len(false_positives) > 0:
    print(false_positives[['Recency', 'Frequency', 'Probability']].describe())